In [0]:
# 01_bronze_ingest_modified.py
# Notebook: 01_bronze_ingest
# Ejecutar en Databricks (Python)

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# *******************************************************************
# RUTA DE ORIGEN (Lectura): Se ajusta la ruta base para leer desde el Volume de Unity Catalog
# *******************************************************************
local_base = "/Volumes/olist/olist_csv/olist1/"

# *******************************************************************
# RUTA DE DESTINO (Escritura): Volume de Unity Catalog
# Asume que el catálogo es 'olist' y el esquema/base de datos es 'olist_csv'.
# *******************************************************************
bronze_path = "/Volumes/olist/olist_csv/bronce_data/"
CATALOG_NAME = "olist"
SCHEMA_NAME = "olist_csv"
VOLUME_NAME = "bronce_data"

# *******************************************************************
# PASO DE CORRECCIÓN [UC_VOLUME_NOT_FOUND]: Crear el Volume si no existe
# Esto requiere permisos de CREAR VOLUME en el esquema/base de datos.
# *******************************************************************
try:
    print(f"Verificando y creando el Volume de destino: {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")
    print("Volume verificado/creado exitosamente.")
except Exception as e:
    print(f"ERROR: No se pudo crear el Volume. Verifica tus permisos de Unity Catalog.")
    print(f"Detalle del error de creación: {e}")
    # Si la creación falla, el script podría fallar después en la escritura.

# *******************************************************************
# MODIFICACIÓN: Lista de archivos corregida para coincidir con la imagen.
# *******************************************************************
files = {
    # Archivos principales del dataset Olist
    "customers": local_base + "olist_customers_dataset.csv",
    "orders": local_base + "olist_orders_dataset.csv",
    "order_items": local_base + "olist_order_items_dataset.csv",
    "order_payments": local_base + "olist_order_payments_dataset.csv",
    "order_reviews": local_base + "olist_order_reviews_dataset.csv",
    "products": local_base + "olist_products_dataset.csv",
    "sellers": local_base + "olist_sellers_dataset.csv",

    # Archivos adicionales encontrados en la imagen
    "geolocation": local_base + "olist_geolocation_dataset.csv",
    "translation": local_base + "product_category_name_translation.csv",
    "premium_flag": local_base + "clientes_premium_flag.csv",
}

# Leer y escribir como parquet (bronze)
for name, path in files.items():
    print(f"Procesando {name} desde {path} ...")
    try:
        # Se mantienen las opciones de CSV: cabecera y inferencia de esquema
        df = spark.read.option("header", True).option("inferSchema", True).csv(path)
        
        # Ruta de salida (en el nuevo Volume)
        out = bronze_path + name
        
        # Se escribe la tabla en formato Parquet en la capa Bronze
        df.write.mode("overwrite").parquet(out)
        print(f" -> Guardado en {out}")
        
    except Exception as e:
        print(f"ERROR: Fallo al procesar o escribir el archivo '{name}'.")
        print(f"Detalle del error: {e}")
        # Detalle para el usuario: Se ha cambiado la ruta de destino a Unity Catalog. 
        # Si el error persiste, verifica que el Volume '{bronze_path}' exista y que tu cluster tenga permisos de ESCRITURA.
        
print("Ingestión Bronze completada.")
